In [0]:
from pyspark import pipelines as dp

@dp.materialized_view(
    name="electroniz_product_highest_grossing_products",
    comment="Finance domain gold-layer materialized view that identifies the highest grossing products by aggregating total sales revenue across in-store and e-commerce channels. Combines data from silver-layer store orders, e-commerce transactions, and the product catalog to provide a unified product revenue ranking for financial analysis.",
    table_properties={
        "domain": "finance",
        "data_product_name": "highest grossing products"
    },
    schema="""
        product_name STRING COMMENT 'The canonical product name from the product catalog, identifying the product sold across in-store and e-commerce sales channels.',
        total_sales DOUBLE COMMENT 'The cumulative gross sales revenue for the product, aggregated from both in-store orders (electroniz_silver_store_orders) and e-commerce transactions (electroniz_silver_ecomm_transactions).'
    """
)
def highest_grossing_products():
    return spark.sql("""
        SELECT
            p.product_name,
            SUM(combined_sales.sale_price) AS total_sales
        FROM (
            SELECT CAST(so.product_id AS STRING) AS product_code, so.sale_price
            FROM electroniz_catalog.electroniz_silver_schema.electroniz_silver_store_orders so
            UNION ALL
            SELECT et.product_code, et.sale_price
            FROM electroniz_catalog.electroniz_silver_schema.electroniz_silver_ecomm_transactions et
        ) combined_sales
        JOIN electroniz_catalog.electroniz_silver_schema.electroniz_silver_products p
            ON combined_sales.product_code = p.product_code
        GROUP BY p.product_name
        ORDER BY total_sales DESC
    """)